
# Building RAG Applications on Databricks: Managed AI Search, Hosted Models & MLflow

## Overview

This demonstration explores how Databricks brings together data management, AI-powered retrieval, model serving, and experiment tracking to build and monitor a Retrieval-Augmented Generation (RAG) application.

We will build a documentation-based AI assistant using Databricks AI Search and hosted foundation models, then use MLflow to trace and evaluate its responses.

## What We Will Demonstrate

**1. Document Processing and Data Management**
- Load and process Databricks documentation.
- Split documents into manageable chunks.
- Store the processed data in Unity Catalog using Delta tables.

**2. Retrieval-Augmented Generation**
- Create a Databricks AI Search index for semantic retrieval.
- Retrieve relevant documentation based on user questions.
- Use Databricks-hosted embedding and chat models to generate context-grounded answers with source references.

**3. MLflow: Observability and Evaluation**
- Enable automatic tracing through LangChain autologging.
- Inspect application execution, model calls, and traces.
- Evaluate response quality using MLflow GenAI evaluation scorers.
- Track evaluation results and store reports as MLflow artifacts.

## Application Architecture

Documentation → Document Processing → Unity Catalog (Delta Table) → Databricks AI Search → Context Retrieval → ChatDatabricks → Grounded Response

**MLflow provides the observability and evaluation layer across the application.**


## 1. Install libraries

- Install LangChain integrations for Databricks and document loading.
- Install the Databricks AI Search SDK.
- Restart Python after installation.

In [0]:
%pip install -U databricks-langchain databricks-ai-search langchain-community langchain-text-splitters beautifulsoup4 mlflow -q

In [0]:
dbutils.library.restartPython()

## 2. Configure the demo

- Use the Unity Catalog three-level namespace: `catalog.schema.table`.
- The catalog and schema must already exist.
- Replace endpoint names with Model Serving endpoints available in your workspace.
- The embedding endpoint must serve a Databricks embedding model.
- The chat endpoint must serve a Databricks chat/instruct model.
- Create an AI Search endpoint in your workspace before running the index section.

In [0]:
%sql 
CREATE CATALOG IF NOT EXISTS dev;
CREATE SCHEMA IF NOT EXISTS dev.genai;

In [0]:
# Unity Catalog three-level namespace
catalog_name = "dev"          
schema_name = "genai"
chunks_table = f"{catalog_name}.{schema_name}.web_rag_chunks"

# Databricks Model Serving endpoints 
embedding_endpoint = "databricks-gte-large-en"
chat_endpoint = "databricks-meta-llama-3-3-70b-instruct"

# AI Search endpoint 
ai_search_endpoint = "rag_search"
vector_index_name = f"{catalog_name}.{schema_name}.web_rag_chunks_index"

source_url = "https://docs.databricks.com/aws/en/ai-search/ai-search"


## 3. Loading the Documentation

A document loader helps retrieve content from a source and make it available for further processing in a RAG application.

- Use LangChain's `WebBaseLoader` to load the selected Databricks documentation page.
- The loader extracts the page content and its source information.
- The extracted content is used in the next step to prepare documents for retrieval.

**Key takeaway:** Document loaders help bring information from external sources into the RAG pipeline.


In [0]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=[source_url])
documents = loader.load()

print("Documents loaded:", len(documents))
print("Metadata:", documents[0].metadata if documents else "No documents returned")
print(documents[0].page_content[:2500] if documents else "No page content returned")


## 4. Splitting Documents into Chunks

- Split the loaded documentation into smaller, manageable text sections called *chunks*.
- Use `RecursiveCharacterTextSplitter`, which prioritizes paragraph and line boundaries before splitting into smaller sections.
- Maintain overlap between consecutive chunks to preserve context that might otherwise be lost at chunk boundaries.

**Key takeaway:** Chunking makes large documents easier to process and helps retrieve relevant information more effectively.

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\\n\\n", "\\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print("Chunks created:", len(chunks))
print(chunks[0].page_content[:1000] if chunks else "No chunks created")

## 5. Save chunks in Unity Catalog

- Create a stable ID for each chunk.
- Keep the text and original source URL as separate columns.
- Save the chunks as a Delta table using the three-level table name.

In [0]:
from pyspark.sql import functions as F

rows = [
    (f"chunk-{i:05d}", i, doc.page_content, doc.metadata.get("source", source_url))
    for i, doc in enumerate(chunks)
]

chunks_df = spark.createDataFrame(
    rows,
    schema="chunk_id STRING, chunk_number INT, chunk_text STRING, chunk_url STRING"
)

(chunks_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(chunks_table))

display(spark.table(chunks_table))

## 6. Enable Change Data Feed

- Delta Sync indexes use the Delta table as their source.
- Enable Change Data Feed for a standard AI Search endpoint.
- This supports incremental synchronization when source data changes.

In [0]:
spark.sql(f"ALTER TABLE {chunks_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")


## 7. Initialize Databricks Embeddings and ChatDatabricks

**Databricks LangChain Integration:**
- `databricks-langchain` provides integrations that allow LangChain applications to work with Databricks-hosted AI models.
- It provides interfaces such as `DatabricksEmbeddings` and `ChatDatabricks` to connect with embedding and chat model endpoints.
- **DatabricksEmbeddings:** Converts text into numerical vector representations (embeddings) using a Databricks-hosted embedding model.
- **ChatDatabricks:** Connects to a Databricks-hosted chat model to generate natural language responses.
- AI Search uses the configured embedding endpoint to generate embeddings for document chunks and user queries.
- Use the same embedding model for indexing and query embedding to ensure consistent vector representations.

**Key takeaway:** Embeddings enable semantic search, while ChatDatabricks generates responses using the retrieved context.

In [0]:
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings

embeddings = DatabricksEmbeddings(endpoint=embedding_endpoint)

llm = ChatDatabricks(
    endpoint=chat_endpoint,
    temperature=0.1,
    max_tokens=700
)

# Quick endpoint checks
test_vector = embeddings.embed_query("What is Databricks AI Search?")
print("Embedding dimensions:", len(test_vector))

test_response = llm.invoke("Explain RAG in one sentence.")
print(test_response.content)


## 8. Creating the Databricks AI Search Endpoint and Index

**What is an AI Search Endpoint?**
- An endpoint provides the infrastructure through which Databricks AI Search operations are performed.
- It hosts and serves the search index, allowing applications to perform retrieval queries.

**What is an AI Search Index?**
- An index organizes document chunks and their embeddings so that relevant information can be retrieved based on a user's query.
- A Delta Sync index uses a Unity Catalog Delta table as its source and can synchronize changes from that table.

**In this implementation:**
- Create a `STANDARD` AI Search endpoint.
- Create a Delta Sync index using our Unity Catalog chunks table.
- Generate embeddings from the `chunk_text` column using a Databricks embedding model endpoint.
- Use a three-level namespace for the index name.

**Key takeaway:** The endpoint provides the serving infrastructure, while the index stores the searchable representation of our document chunks.

In [0]:
from databricks.ai_search.client import AISearchClient

ai_search_client = AISearchClient(disable_notice=True)

In [0]:
if not ai_search_client.endpoint_exists(ai_search_endpoint):
    print(f"Creating AI Search endpoint: {ai_search_endpoint}")

    ai_search_client.create_endpoint(
        name=ai_search_endpoint,
        endpoint_type="STANDARD"
    )

    ai_search_client.wait_until_endpoint_ready(
        name=ai_search_endpoint
    )

    print(f"AI Search endpoint '{ai_search_endpoint}' created successfully.")
else:
    print(f"AI Search endpoint '{ai_search_endpoint}' already exists.")

if not ai_search_client.index_exists(index_name=vector_index_name):
    index = ai_search_client.create_delta_sync_index(
        endpoint_name=ai_search_endpoint,
        source_table_name=chunks_table,
        index_name=vector_index_name,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=embedding_endpoint
    )
else:
    index = ai_search_client.get_index(index_name=vector_index_name)

print("AI Search index:", vector_index_name)

## 9. Sync and wait for the index

- Trigger the index pipeline to ingest the latest Delta table contents.
- Wait until the index is ready before running retrieval.

In [0]:
index.sync()
index.wait_until_ready()
display(index.describe())

## 10. Retrieve relevant chunks

- Send the question to AI Search as text.
- AI Search uses the configured Databricks embedding endpoint to embed the query.
- Return the top-K most relevant chunks and their source URLs.

In [0]:
def retrieve_chunks(question: str, k: int = 4):
    return index.similarity_search(
        query_text=question,
        columns=["chunk_id", "chunk_text", "chunk_url"],
        num_results=k
    )

question = "How does hybrid search work in Databricks AI Search?"
retrieval_result = retrieve_chunks(question)
retrieval_result

## 11. Build context from retrieved chunks

- Extract the retrieved text and source URL from the AI Search response.
- Combine the chunks into one context block for the LLM.

In [0]:
def format_context(search_result):
    manifest = search_result.get("manifest", {})
    columns = manifest.get("columns", [])
    rows = search_result.get("result", {}).get("data_array", [])

    column_names = [column.get("name") for column in columns]
    formatted_chunks = []

    for row in rows:
        record = dict(zip(column_names, row))
        formatted_chunks.append(
            f"Source: {record.get('chunk_url')}\\n"
            f"Content: {record.get('chunk_text')}"
        )

    return "\\n\\n---\\n\\n".join(formatted_chunks)

context = format_context(retrieval_result)
print(context[:4000])

## 12. Generate a grounded RAG answer

- Pass the retrieved context and question to ChatDatabricks.
- Instruct the model to answer only from the retrieved content.
- Ask it to cite source URLs and admit when the context is insufficient.

In [0]:
from langchain_core.prompts import ChatPromptTemplate
from databricks_langchain import ChatDatabricks
def run_rag(question):
    rag_prompt = ChatPromptTemplate.from_messages([
        ("system",
        "You are a Databricks documentation assistant. "
        "Answer only using the context provided. "
        "If the context does not contain the answer, say: "
        "'I could not find the answer in the retrieved documentation.' "
        "Do not invent details. Cite the source URL for factual claims.\\n\\n"
        "Context:\\n{context}"),
        ("human", "{question}")
    ])
    
    retrieval_result = retrieve_chunks(question)

    context = format_context(retrieval_result)

    llm = ChatDatabricks(
        model = chat_endpoint,
        temprature = 0.1,
        max_tokens = 2000
    )
    rag_chain = rag_prompt | llm

    answer = rag_chain.invoke({"context": context, "question": question})
    return answer.content
print(run_rag("how does databricks ai search work?"))

## 13. MLflow: Experiment Tracking and Tracing

- MLflow helps monitor, evaluate, and track GenAI applications.
- Experiments organize related development and evaluation runs.
- Tracing captures the execution flow of an application, including inputs, outputs, model calls, and latency.
- LangChain autologging automatically captures supported LangChain operations.
- Traces help identify execution issues and understand application behavior.


In [0]:

import mlflow

# Set the MLflow experiment
mlflow.set_experiment("/Shared/databricks-rag-demo")

# Enable automatic tracing for supported LangChain operations
mlflow.langchain.autolog()

# Execute the RAG application to generate a trace
traced_answer = run_rag("What is an AI search index?")

print(traced_answer)


### Inspecting MLflow Traces

- Every traced execution can be inspected through the MLflow experiment UI.
- Review the input question and generated response.
- Inspect supported LangChain operations and model calls.
- Analyze execution duration and identify errors.
- Tracing provides visibility into the internal execution of the RAG application.


## 14. GenAI Evaluation

- Tracing helps us understand how the application executes, while evaluation helps measure its output quality.
- MLflow GenAI Evaluation provides a framework to assess LLM applications using evaluation datasets and scorers.
- An evaluation dataset contains input questions and their expected responses.
- Scorers assess generated answers based on defined quality criteria.
- Evaluation results help identify areas where the RAG application can be improved.


### Preparing the Evaluation Dataset

- Create a small set of representative questions from the source documentation.
- Define an expected response for each question.
- Include different question types to test the application's understanding:
  - Conceptual questions
  - Technical understanding
  - Comparison and differentiation questions
- Use this dataset to evaluate the RAG application's generated answers.
- The expected responses act as reference answers for correctness evaluation.


In [0]:
evaluation_data = [
    {
        "inputs": {
            "question": "What is hybrid search?"
        },
        "expectations": {
            "expected_response": (
                "Hybrid search combines vector search and keyword search "
                "to retrieve relevant results."
            )
        }
    },
    {
        "inputs": {
            "question": "What is the role of embeddings in AI Search?"
        },
        "expectations": {
            "expected_response": (
                "Embeddings represent text as numerical vectors. "
                "AI Search uses them to find semantically similar content."
            )
        }
    },
    {
        "inputs": {
            "question": "What is the difference between vector search and hybrid search?"
        },
        "expectations": {
            "expected_response": (
                "Vector search uses embeddings to find documents based on "
                "semantic similarity, while hybrid search combines vector "
                "search with keyword-based search (BM25) to retrieve results. "
                "Hybrid search benefits from both semantic understanding "
                "and exact keyword matching."
            )
        }
    }
]

### Configuring MLflow Evaluation

- Use MLflow GenAI scorers to evaluate the generated responses.
- Correctness checks whether the answer matches the expected answer.
- RelevanceToQuery checks whether the answer addresses the user's question.
- MLflow records the evaluation results and associated traces for inspection.


In [0]:

# Enable system metrics collection for MLflow runs
mlflow.enable_system_metrics_logging()

# Set sampling interval to 2 seconds for the demo
mlflow.set_system_metrics_sampling_interval(2)


In [0]:

from mlflow.genai.scorers import (
    Correctness,
    RelevanceToQuery
)

# MLflow passes each question to this function
def predict_fn(question: str) -> str:
    response = run_rag(question)
    return response

# Run evaluation
evaluation_results = mlflow.genai.evaluate(
    data=evaluation_data,
    predict_fn=predict_fn,
    scorers=[
        Correctness(),
        RelevanceToQuery()
    ]
)

print("Evaluation completed!")
print("Evaluation Run ID:", evaluation_results.run_id)

print("\nAggregate Metrics:")
print(evaluation_results.metrics)


### Reviewing Evaluation Results

- Aggregate metrics provide an overview of application quality.
- Per-question results show how individual responses performed.
- Scorer assessments provide feedback on answer correctness and relevance.
- Evaluation traces help investigate individual predictions.


In [0]:

# Display detailed evaluation results for each question
results_df = evaluation_results.result_df

display(results_df)


## 15. Logging Evaluation Artifacts

- MLflow allows us to store files associated with an experiment run.
- Evaluation reports can be saved as artifacts for future reference.
- Artifacts make it easier to review and compare evaluation results.
- Examples include evaluation datasets, CSV reports, prompts, and configuration files.


In [0]:

import mlflow

# Save per-question evaluation results
csv_path = "/Workspace/Users/jainanvesha14@gmail.com/Day 13 (RAG,  MLflow, prompting)/rag_evaluation_results.csv"
results_df.to_csv(csv_path, index=False)

# Log the CSV report as an artifact
client = mlflow.MlflowClient()

client.log_artifact(
    evaluation_results.run_id,
    csv_path,
    artifact_path="evaluation"
)

print("Evaluation report logged successfully!")



## MLflow in Databricks vs. Self-Managed MLflow

MLflow is **open source** — it can be installed and run completely independently of Databricks, on your own laptop, server, or any cloud.

The tracking API you use in code is identical either way:
`mlflow.set_experiment()`, `mlflow.langchain.autolog()`, `mlflow.genai.evaluate()` — same calls, same behavior.

What differs is the infrastructure and governance around it:

| | Self-managed MLflow | Managed MLflow on Databricks |
|---|---|---|
| Tracking server | You host and maintain it yourself | Fully managed, already running |
| Storage for traces/results | You configure your own backend store + artifact store (e.g., your own database + S3 bucket) | Stored automatically as governed Delta tables in Unity Catalog, no per-experiment storage limit |
| Access control / governance | You build this yourself | Built-in, via Unity Catalog's existing permission model |
| Exploring results | Basic built-in UI only | Full UI + ask questions in plain English via Genie Code |
| Scaling, uptime, backups | Your responsibility | Handled by Databricks |

**Takeaway:** it isn't different tooling — it's the same open-source MLflow, with the operational overhead of running it yourself removed. Same theme as the AI Search Index/Endpoint earlier: Databricks manages the infrastructure so you can focus on the application.

## 16. Demo Wrap-up

### What We Demonstrated

- **RAG:** Grounds LLM responses in retrieved documentation.
- **Model Serving:** Provides managed embedding and chat model endpoints.
- **Unity Catalog:** Stores and governs the source data.
- **Databricks AI Search:** Provides managed semantic retrieval.
- **MLflow Tracing:** Provides visibility into application execution.
- **MLflow Evaluation:** Measures answer quality using evaluation scorers.
- **Experiment Tracking:** Stores run information, metrics, and artifacts.

**Presentation Takeaway:**

Databricks brings data governance, model serving, retrieval, and GenAI observability and evaluation together in a unified platform.
